# 03. panel

## 0. setup

In [1]:
import gc
from pathlib import Path
 
import numpy as np
import pandas as pd

# paths
root         = Path.cwd().parent
data_raw     = root / 'data' / 'raw'
data_interim = root / 'data' / 'interim'
data_proc    = root / 'data' / 'processed'

data_proc.mkdir(parents=True, exist_ok=True)

## 1. config

In [ ]:
min_cell_pat = 10               # a cell must accumulate >= this many fractional patents
                                # over the window to enter the panel (drops non-innovative cells)

in_treat  = data_proc    / 'cartel_treated.parquet'   # infr x nace3 x ctry
in_pat    = data_interim / 'pat_panel.parquet'        # ctry x isic3 x year
out_panel = data_proc    / 'panel.parquet'

## 2. treated cells to industry grain (single first-treat collapse)

In [3]:
treat = pd.read_parquet(in_treat)
treat = treat.rename(columns={'ctry_iso2': 'ctry_iso'})

treat_cell = (treat.groupby(['ind', 'ctry_iso'])
              .agg(cohort_decision=('cohort_decision', 'min'),
                   cohort_start   =('cohort_start', 'min'),
                   cohort_end     =('cohort_end', 'min'),
                   event_ids      =('enforcement_event_id',
                                    lambda s: sorted(set(s.dropna()))))
              .reset_index())
treat_cell['n_events']       = treat_cell['event_ids'].apply(len)
treat_cell['repeat_treated'] = (treat_cell['n_events'] > 1).astype(int)

print(f'treated cells at ISIC3: {len(treat_cell)} '
      f'| industries: {treat_cell['ind'].nunique()} '
      f'| repeat-treated: {treat_cell['repeat_treated'].sum()}')

treated cells at ISIC3: 276 | industries: 34 | repeat-treated: 51


## 3. balanced grid + zero-filled outcomes

In [4]:
pat = pd.read_parquet(in_pat)

# cell inclusion: enough cumulative patent activity to be a meaningful innovation cell
cell_tot = pat.groupby(['ind', 'ctry_iso'])['pat_frac'].sum().rename('cell_tot').reset_index()
keep_cells = cell_tot[cell_tot['cell_tot'] >= min_cell_pat][['ind', 'ctry_iso']]

# ensure treated cells that clear the floor are represented even if patent-thin
keep_cells = pd.concat([keep_cells, treat_cell[['ind', 'ctry_iso']]], ignore_index=True).drop_duplicates()
keep_cells = keep_cells.merge(cell_tot, on=['ind', 'ctry_iso'], how='left')
keep_cells = keep_cells[keep_cells['cell_tot'].fillna(0) >= min_cell_pat][['ind', 'ctry_iso']]

# balanced (ind x ctry) x year grid
y_lo, y_hi = int(pat['year'].min()), int(pat['year'].max())
yrs = pd.DataFrame({'year': range(y_lo, y_hi + 1)})
panel = keep_cells.merge(yrs, how='cross')


# join outcomes; true zeros where a cell-year had no patents
panel = panel.merge(pat, on=['ind', 'ctry_iso', 'year'], how='left')
for col in ['pat_frac', 'n_pat_appln', 'n_applt']:
    panel[col] = panel[col].fillna(0)

print(f'panel grid: {len(panel):,} cell-years | cells: {len(keep_cells):,} | years {y_lo}-{y_hi}')

panel grid: 142,222 cell-years | cells: 3,026 | years 1978-2024


## 4. merge treatment + event-time (three designs)
`evt_dec` (decision date), `evt_form` (formation), `evt_brk` (breakup) carried as parallel DiD designs.

In [5]:
panel = panel.merge(treat_cell, on=['ind', 'ctry_iso'], how='left')
panel['ever_treated']   = panel['cohort_decision'].notna().astype(int)
panel['repeat_treated'] = panel['repeat_treated'].fillna(0).astype(int)

# per design: event time and absorbing post indicator
designs = {'dec': 'cohort_decision', 'form': 'cohort_start', 'brk': 'cohort_end'}
for tag, coh in designs.items():
    panel[f'evt_{tag}']   = panel['year'] - panel[coh]
    panel[f'treat_{tag}'] = ((panel[coh].notna()) & (panel['year'] >= panel[coh])).astype(int)

# ISIC division for division-level FE / clustering specs
panel['isic2'] = panel['ind'].str[:2]

# outcome transforms (count is sparse/skewed)
panel['ln1p_pat']  = np.log1p(panel['pat_frac'])
panel['asinh_pat'] = np.arcsinh(panel['pat_frac'])

panel = panel.sort_values(['ind', 'ctry_iso', 'year']).reset_index(drop=True)
print('columns:', list(panel.columns))

columns: ['ind', 'ctry_iso', 'year', 'pat_frac', 'n_pat_appln', 'n_applt', 'cohort_decision', 'cohort_start', 'cohort_end', 'event_ids', 'n_events', 'repeat_treated', 'ever_treated', 'evt_dec', 'treat_dec', 'evt_form', 'treat_form', 'evt_brk', 'treat_brk', 'isic2', 'ln1p_pat', 'asinh_pat']


## 5. coverage of treated industries at this grain

In [7]:
treated_ind = set(treat_cell['ind'])
panel_ind   = set(panel['ind'])
have        = treated_ind & panel_ind
treated_cells_all = set(map(tuple, treat_cell[['ind','ctry_iso']].values))
treated_cells_in  = set(map(tuple, panel.loc[panel['ever_treated']==1, ['ind','ctry_iso']]
                                     .drop_duplicates().values))

print(f'treated industries (ISIC3)            : {len(treated_ind)}')
print(f'  with a patent-outcome cell in panel : {len(have)}')
print(f'  no patent outcome (dropped)         : {sorted(treated_ind - panel_ind)}')
print(f'treated cells                         : {len(treated_cells_all)}')
print(f'  surviving min_cell_pat              : {len(treated_cells_in)}   (effective treated N)')
print(f'  dropped by floor                    : {len(treated_cells_all - treated_cells_in)}')

treated industries (ISIC3)            : 34
  with a patent-outcome cell in panel : 26
  no patent outcome (dropped)         : ['204', '234', '236', '244', '462', '463', '491', '502']
treated cells                         : 276
  surviving min_cell_pat              : 178   (effective treated N)
  dropped by floor                    : 98


## 6. diagnostics

In [9]:
n_cells   = panel[['ind', 'ctry_iso']].drop_duplicates().shape[0]
n_treated = panel.loc[panel['ever_treated'] == 1, ['ind', 'ctry_iso']].drop_duplicates().shape[0]
n_tr_ind  = panel.loc[panel['ever_treated'] == 1, 'ind'].nunique()

print(f'cells            : {n_cells:,}')
print(f'  treated cells  : {n_treated}   (this is the effective treated N)')
print(f'  treated industries: {n_tr_ind}')
print(f'  never-treated  : {n_cells - n_treated}')

for tag, coh in [('decision','cohort_decision'), ('start','cohort_start'), ('end','cohort_end')]:
    print(f'\n=== {tag}-cohort sizes (treated cells per cohort year) ===')
    tc = (panel.loc[panel['ever_treated'] == 1, ['ind', 'ctry_iso', coh]].drop_duplicates())
    cs = tc.groupby(coh).size()
    print(cs.to_string())
    print('singleton cohorts:', list(cs[cs == 1].index))

print('\n=== event-time coverage (decision design, treated cells) ===')
evt = panel.loc[panel['ever_treated'] == 1, 'evt_dec']
print(evt.value_counts().sort_index().to_string())

print('\n=== patents ===')
print(panel[['pat_frac', 'ln1p_pat', 'asinh_pat']].describe().round(3).to_string())
print('share of cell-years with zero patents:', f'{(panel['pat_frac']==0).mean():.1%}')

cells            : 3,026
  treated cells  : 178   (this is the effective treated N)
  treated industries: 24
  never-treated  : 2848

=== decision-cohort sizes (treated cells per cohort year) ===
cohort_decision
1994     6
1998    17
1999     2
2000     1
2001     2
2002     3
2004     2
2005    17
2007     8
2008    19
2009    34
2010    18
2011     1
2012     1
2013    21
2014    10
2015     9
2017     3
2021     3
2022     1
singleton cohorts: [np.int64(2000), np.int64(2011), np.int64(2012), np.int64(2022)]

=== start-cohort sizes (treated cells per cohort year) ===
cohort_start
1980     2
1982     6
1984    13
1985     1
1986     3
1987     1
1988     6
1989     1
1990    18
1991     1
1992     8
1993     4
1994    11
1995     3
1996     4
1998     7
1999    16
2000     9
2002     4
2004    42
2005    10
2006     1
2008     3
2009     3
2011     1
singleton cohorts: [np.int64(1985), np.int64(1987), np.int64(1989), np.int64(1991), np.int64(2006), np.int64(2011)]

=== end-cohort size

## 7. save

In [10]:
panel_out = panel.copy()
panel_out['event_ids'] = panel_out['event_ids'].apply(
    lambda x: ','.join(map(str, x)) if isinstance(x, (list, tuple, np.ndarray)) else '')
panel_out.to_parquet(out_panel, index=False)
print(f'saved: {out_panel.name} ({len(panel_out):,} rows)')

saved: panel.parquet (142,222 rows)
